In [0]:
WITH tb_pedidos AS (
  SELECT *
  FROM workspace.olist.orders
  WHERE order_purchase_timestamp < '2018-07-01'
),
tb_data AS (
    SELECT '2018-07-01' as dt_referencia
),
tb_seller_rfv AS (
    select  
        s.seller_id,
        count(distinct p.order_id) as qtde_pedidos_vida,
        count(distinct case when datediff(d.dt_referencia, p.order_purchase_timestamp) <= 28 then p.order_id end) as qtde_pedidos_d28,
        count(distinct case when datediff(d.dt_referencia, p.order_purchase_timestamp) <= 56 then p.order_id end) as qtde_pedidos_d56,
        count(distinct case when datediff(d.dt_referencia, p.order_purchase_timestamp) <= 365 then p.order_id end) as qtde_pedidos_d365,
        count(distinct case when order_status = 'delivered' then p.order_id end) as qtde_pedidos_delivered_vida,
        count(distinct case when order_status = 'invoiced' then p.order_id end) as qtde_pedidos_invoiced_vida,
        count(distinct case when order_status = 'shipped' then p.order_id end) as qtde_pedidos_shipped_vida,
        count(distinct case when order_status = 'processing' then p.order_id end) as qtde_pedidos_processing_vida,
        count(distinct case when order_status = 'unavailable' then p.order_id end) as qtde_pedidos_unavailable_vida,
        count(distinct case when order_status = 'canceled' then p.order_id end) as qtde_pedidos_canceled_vida,
        count(distinct case when order_status = 'created' then p.order_id end) as qtde_pedidos_created_vida,
        count(distinct case when order_status = 'approved' then p.order_id end) as qtde_pedidos_approved_vida                           
    from tb_pedidos p
    left join workspace.olist.order_items i on p.order_id = i.order_id
    left join workspace.olist.sellers s on i.seller_id = s.seller_id
    cross join tb_data d
    group by s.seller_id 
)
SELECT * FROM tb_seller_rfv